# Four-Format Data Recovery and Validation

This notebook imports and runs the project pipeline, then summarizes recovery, cross-format agreement, exports, and analyses. The case study is a historical 1952–2007 teaching snapshot; population is measured in persons, and life expectancy in years. It is not a description of current conditions.

Run from the repository root after installing `requirements.txt`.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

from src.analysis import (
    compute_independent_asian_total,
    compute_numpy_population_matrix,
)
from src.pipeline import run_pipeline

ROOT = Path.cwd()
CONFIG_PATH = ROOT / "config.json"
OUTPUT_DIR = ROOT / "outputs"
MANIFEST = json.loads((ROOT / "data" / "manifest.json").read_text(encoding="utf-8"))
CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
REPORTING_YEAR = CONFIG.get("reporting_year", 2007)

assert run_pipeline(CONFIG_PATH) == 0, "The configured recovery pipeline failed."
print(f"Pipeline completed for reporting year {REPORTING_YEAR}.")

Processing complete: 1704 canonical rows; outputs published to C:\Users\jdpad\Documents\Week-10-Group-Programming-Challenge\Week-10-Group-Programming-Challenge-Four-Format-Data-Recovery-and-Validation\outputs
Pipeline completed for reporting year 2007.


## Recovery and validation

Each source has 1,712 records. The expected per-file reconciliation is 1,712 raw = 6 quarantined + 2 exact duplicate copies + 1,704 canonical rows. The canonical table is one recovered dataset, not four concatenated copies.

In [2]:
canonical = pd.read_csv(OUTPUT_DIR / "canonical.csv")
quarantine = pd.read_csv(OUTPUT_DIR / "quarantine.csv")
duplicates = pd.read_csv(OUTPUT_DIR / "duplicate_log.csv")
validation = pd.read_csv(OUTPUT_DIR / "validation.csv")
audit = pd.read_csv(OUTPUT_DIR / "audit_log.csv")

recovery_summary = pd.DataFrame(
    [
        ("Raw records across four inputs", len(MANIFEST["files"]) * MANIFEST["expected_per_file"]["raw_rows"]),
        ("Quarantined records", len(quarantine)),
        ("Removed exact duplicate copies", len(duplicates)),
        ("Canonical records", len(canonical)),
        ("Validation checks passed", int(validation["pass"].sum())),
        ("Validation checks total", len(validation)),
    ],
    columns=["Measure", "Value"],
)
display(recovery_summary)
assert validation["pass"].all()
assert list(validation.columns) == ["check", "expected", "actual", "tolerance", "pass"]
assert len(canonical) == MANIFEST["expected_per_file"]["canonical_rows"]

,Measure,Value
0,Raw records across four inputs,6848
1,Quarantined records,24
2,Removed exact duplicate copies,8
3,Canonical records,1704
4,Validation checks passed,80
5,Validation checks total,80


## Agreement and export round trips

The validation report compares each non-reference input with the first recovered table by key. Canonical exports are also reloaded and compared with the pre-export table. Text, keys, integer values, and missingness are exact; floating-point comparisons use absolute tolerance 1e-8 and relative tolerance 0.

In [3]:
agreement_checks = validation.loc[
    validation["check"].str.contains(
        r"country_records\.(xlsx|json|parquet):(row_count|columns|unique_keys|key_set|values:)",
        regex=True,
    )
].copy()
round_trip_checks = validation.loc[
    validation["check"].str.contains(
        r"canonical\.(csv|xlsx|json|parquet):", regex=True
    )
].copy()
print(f"Cross-format agreement: {int(agreement_checks['pass'].sum())}/{len(agreement_checks)} checks passed")
display(agreement_checks[["check", "expected", "actual", "pass"]].head(16))
print(f"Export round trips: {int(round_trip_checks['pass'].sum())}/{len(round_trip_checks)} checks passed")
display(round_trip_checks[["check", "expected", "actual", "tolerance", "pass"]].head(12))
assert agreement_checks["pass"].all()
assert round_trip_checks["pass"].all()

Cross-format agreement: 24/24 checks passed


C:\Users\jdpad\AppData\Local\Temp\ipykernel_8204\588039180.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  validation["check"].str.contains(
C:\Users\jdpad\AppData\Local\Temp\ipykernel_8204\588039180.py:8: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  validation["check"].str.contains(


,check,expected,actual,pass
12,country_records.xlsx:row_count,1704,1704,True
13,country_records.xlsx:columns,"['country', 'continent', 'year', 'lifeExp', 'p...","['country', 'continent', 'year', 'lifeExp', 'p...",True
14,country_records.xlsx:unique_keys,True,True,True
15,country_records.xlsx:key_set,1704,1704,True
16,country_records.xlsx:values:continent,exact,match,True
17,country_records.xlsx:values:lifeExp,absolute tolerance,match,True
18,country_records.xlsx:values:pop,exact,match,True
19,country_records.xlsx:values:gdpPercap,absolute tolerance,match,True
20,country_records.json:row_count,1704,1704,True
21,country_records.json:columns,"['country', 'continent', 'year', 'lifeExp', 'p...","['country', 'continent', 'year', 'lifeExp', 'p...",True


Export round trips: 32/32 checks passed


,check,expected,actual,tolerance,pass
48,canonical.csv:row_count,1704,1704,0.000000e+00,True
49,canonical.csv:columns,"['country', 'continent', 'year', 'lifeExp', 'p...","['country', 'continent', 'year', 'lifeExp', 'p...",0.000000e+00,True
50,canonical.csv:unique_keys,True,True,0.000000e+00,True
51,canonical.csv:key_set,1704,1704,0.000000e+00,True
52,canonical.csv:values:continent,exact,match,0.000000e+00,True
53,canonical.csv:values:lifeExp,absolute tolerance,match,1.000000e-08,True
54,canonical.csv:values:pop,exact,match,0.000000e+00,True
55,canonical.csv:values:gdpPercap,absolute tolerance,match,1.000000e-08,True
56,canonical.xlsx:row_count,1704,1704,0.000000e+00,True
57,canonical.xlsx:columns,"['country', 'continent', 'year', 'lifeExp', 'p...","['country', 'continent', 'year', 'lifeExp', 'p...",0.000000e+00,True


## Pandas and NumPy analysis

The selected-year Asia table uses the configured year, population in millions, and descending life-expectancy ranks with minimum rank for ties. The NumPy population index uses the 1982 value for each country as its 100-point baseline. A Boolean selection identifies countries above 70 million population in 2007 and aggregates their mean population.

In [4]:
asia = pd.read_csv(OUTPUT_DIR / "asia_selected_year.csv")
philippines = pd.read_csv(OUTPUT_DIR / "country_history.csv")
population_index = pd.read_csv(OUTPUT_DIR / "population_index.csv")
matrix = compute_numpy_population_matrix(canonical)

print(f"Asia rows for {REPORTING_YEAR}: {len(asia)}")
display(asia.head(10))
print(f"Philippines observations: {len(philippines)}")
display(philippines[["year", "lifeExp", "pop", "pop_millions", "lifeExp_change"]])
print(f"NumPy shape before broadcasting: {matrix['shape_before']}")
print(f"NumPy shape after broadcasting: {matrix['shape_after']}")
print(f"Selected 2007 countries (>70 million): {matrix['filtered_countries_2007']}")
print(f"Mean selected population: {matrix['mean_pop_large_countries_2007']:,.1f} persons")
display(population_index)

independent_asia_total = compute_independent_asian_total(canonical, REPORTING_YEAR)
print(f"Independent NumPy Asian population total for {REPORTING_YEAR}: {independent_asia_total:,} persons")
if REPORTING_YEAR == 2007:
    assert independent_asia_total == MANIFEST["controls"]["asia_2007_population"]
    philippines_2007 = int(philippines.loc[philippines["year"] == 2007, "pop"].iloc[0])
    assert philippines_2007 == MANIFEST["controls"]["philippines_2007_population"]
    print("The 2007 Asian and Philippine population controls match the manifest.")

Asia rows for 2007: 33


,country,continent,year,lifeExp,pop,gdpPercap,pop_millions,lifeExp_rank
0,Japan,Asia,2007,82.603,127467972,31656.068060,127.467972,1
1,"Hong Kong, China",Asia,2007,82.208,6980412,39724.978670,6.980412,2
2,Israel,Asia,2007,80.745,6426679,25523.277100,6.426679,3
3,Singapore,Asia,2007,79.972,4553009,47143.179640,4.553009,4
4,"Korea, Rep.",Asia,2007,78.623,49044790,23348.139730,49.044790,5
5,Taiwan,Asia,2007,78.400,23174294,28718.276840,23.174294,6
6,Kuwait,Asia,2007,77.588,2505559,47306.989780,2.505559,7
7,Oman,Asia,2007,75.640,3204897,22316.192870,3.204897,8
8,Bahrain,Asia,2007,75.635,708573,29796.048340,0.708573,9
9,Vietnam,Asia,2007,74.249,85262356,2441.576404,85.262356,10


Philippines observations: 12


,year,lifeExp,pop,pop_millions,lifeExp_change
0,1952,47.752,22438691,22.438691,0.000
1,1957,51.334,26072194,26.072194,3.582
2,1962,54.757,30325264,30.325264,7.005
3,1967,56.393,35356600,35.356600,8.641
4,1972,58.065,40850141,40.850141,10.313
5,1977,60.060,46850962,46.850962,12.308
6,1982,62.082,53456774,53.456774,14.330
7,1987,64.151,60017788,60.017788,16.399
8,1992,66.458,67185766,67.185766,18.706
9,1997,68.564,75012988,75.012988,20.812


NumPy shape before broadcasting: (3, 6)
NumPy shape after broadcasting: (3, 6)
Selected 2007 countries (>70 million): ['Philippines', 'Indonesia']
Mean selected population: 157,312,143.5 persons


,country,1982,1987,1992,1997,2002,2007
0,Philippines,100.0,112.273494,125.682418,140.324570,155.256447,170.375577
1,Indonesia,100.0,110.390432,120.524576,129.955720,137.639149,145.782331
2,Thailand,100.0,108.362522,116.056504,123.326192,128.630762,133.262203


Independent NumPy Asian population total for 2007: 3,811,953,827 persons
The 2007 Asian and Philippine population controls match the manifest.


## Interpretation

The following summary is generated from the recovered tables. Population totals and population-index values are in persons and index points respectively; the historical dataset should not be interpreted as current population or life-expectancy estimates.

In [5]:
best = asia.iloc[0]
first_ph = philippines.iloc[0]
last_ph = philippines.iloc[-1]
print(
    f"For {REPORTING_YEAR}, the Asian observations include {len(asia)} countries. "
    f"The highest life-expectancy rank is held by {best['country']} "
    f"({best['lifeExp']:.2f} years; rank {int(best['lifeExp_rank'])})."
)
print(
    f"The Philippine series contains {len(philippines)} observations from "
    f"{int(first_ph['year'])} to {int(last_ph['year'])}; life expectancy changes "
    f"from {first_ph['lifeExp']:.2f} to {last_ph['lifeExp']:.2f} years "
    f"({last_ph['lifeExp_change']:+.2f} years from its first observation)."
)
print(
    f"All {len(validation)} validation checks pass. The four canonical "
    "exports also pass their key-based round-trip checks."
)

For 2007, the Asian observations include 33 countries. The highest life-expectancy rank is held by Japan (82.60 years; rank 1).
The Philippine series contains 12 observations from 1952 to 2007; life expectancy changes from 47.75 to 71.69 years (+23.94 years from its first observation).
All 80 validation checks pass. The four canonical exports also pass their key-based round-trip checks.
